# EP-133 Mobile Studio 4.3.4 Beta · Hardened Build v5 — исправленная облачная сборка APK

Этот блокнот собирает **debug APK** в Google Colab. Android Studio и программирование не нужны.

1. Загрузите `Zvukotseh_Studio_Pro_1.0.0_Source.zip` из комплекта релиза.
2. Откройте блокнот в Google Colab и выберите **Runtime → Run all**.
3. Загрузите ZIP исходников, когда появится окно. Блокнот сверит SHA-256 и безопасно распакует архив.
4. Затем установит инструменты, проверит исходники и выполнит `clean assembleDebug`.
5. При успехе Colab проверит APK и скачает `Zvukotseh_Studio_Pro_1.0.0-debug.apk`.

v5 исправляет передачу подтверждений лицензий Android SDK: теперь отправляются настоящие строки перевода строки, а не буквальные символы `\\n`. Остальные проверки ZIP, SHA-256, APK и метаданных сохранены. Минимум Android 9 (API 28). Физическое тестирование на устройствах обязательно.


In [ ]:
import os, glob, zipfile, subprocess, shutil, hashlib, pathlib, re, stat
from google.colab import files

EXPECTED_SHA256 = "6b67a0d5912ececbddce986df24b08dc7422c2936706ee37253022cebc8e92e1"
ANDROID_TOOLS_SHA256 = "2d2d50857e4eb553af5a6dc3ad507a17adf43d115264b1afc116f95c92e5e258"
GRADLE_SHA256 = "d725d707bfabd4dfdc958c624003b3c80accc03f7037b5122c4b1d0ef15cecab"

def verify_sha256(path, expected, label):
    actual = hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"SHA-256 не совпадает для {label}: ожидался {expected}, получен {actual}")
    print(f"SHA-256 подтверждён: {label}")

def safe_extract(zip_path, destination):
    destination = pathlib.Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    if any(destination.iterdir()):
        raise RuntimeError("Папка назначения должна быть пустой")

    max_entries = 512
    max_total_uncompressed = 256 * 1024 * 1024
    max_single_file = 96 * 1024 * 1024
    max_compression_ratio = 1000
    seen = set()
    file_paths = set()
    total_declared = 0

    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        if not members or len(members) > max_entries:
            raise RuntimeError(f"Недопустимое число файлов в архиве: {len(members)}")

        # Validate the entire directory tree and declared sizes BEFORE decompressing anything.
        for member in members:
            raw_name = member.filename
            if not raw_name or "\\" in raw_name or "\x00" in raw_name:
                raise RuntimeError("Недопустимое имя файла в ZIP")
            # PurePosixPath normalizes away "." and repeated separators, so inspect
            # the raw path first. A trailing slash is allowed only for directory entries.
            raw_parts = raw_name.split("/")
            is_directory_name = raw_name.endswith("/")
            checked_parts = raw_parts[:-1] if is_directory_name else raw_parts
            if (raw_name.startswith("/") or not checked_parts or
                    any(part in ("", ".", "..") for part in checked_parts)):
                raise RuntimeError("Небезопасный путь в архиве: " + raw_name)
            name = pathlib.PurePosixPath(raw_name)
            if name.is_absolute():
                raise RuntimeError("Небезопасный путь в архиве: " + raw_name)
            if name.parts and ":" in name.parts[0]:
                raise RuntimeError("Недопустимый путь Windows в архиве: " + raw_name)

            normalized = name.as_posix().rstrip("/")
            if not normalized or normalized in seen:
                raise RuntimeError("Пустой или повторяющийся путь в архиве: " + raw_name)
            seen.add(normalized)

            mode = (member.external_attr >> 16) & 0xFFFF
            if stat.S_ISLNK(mode):
                raise RuntimeError("Символические ссылки в ZIP запрещены: " + raw_name)
            file_type = stat.S_IFMT(mode)
            if file_type and file_type not in (stat.S_IFREG, stat.S_IFDIR):
                raise RuntimeError("Специальные файлы в ZIP запрещены: " + raw_name)
            if member.flag_bits & 0x1:
                raise RuntimeError("Зашифрованные ZIP-записи не поддерживаются")

            is_dir = member.is_dir()
            if is_directory_name != is_dir:
                raise RuntimeError("Имя пути и тип ZIP-записи не совпадают: " + raw_name)
            if is_dir and member.file_size != 0:
                raise RuntimeError("Каталог с данными в ZIP: " + raw_name)
            if not is_dir:
                if member.file_size > max_single_file:
                    raise RuntimeError("Слишком большой файл в архиве: " + raw_name)
                if member.file_size and member.compress_size == 0:
                    raise RuntimeError("Некорректный размер сжатого файла: " + raw_name)
                if member.compress_size and member.file_size / member.compress_size > max_compression_ratio:
                    raise RuntimeError("Подозрительная степень сжатия ZIP: " + raw_name)
                total_declared += member.file_size
                if total_declared > max_total_uncompressed:
                    raise RuntimeError("Суммарный распакованный размер ZIP превышает лимит")
                file_paths.add(normalized)

            target = (destination / pathlib.Path(*name.parts)).resolve()
            if os.path.commonpath([str(destination), str(target)]) != str(destination):
                raise RuntimeError("Небезопасный путь в архиве: " + raw_name)

        # A regular file cannot also be a parent directory of another archive entry.
        for path in seen:
            parts = pathlib.PurePosixPath(path).parts
            for i in range(1, len(parts)):
                if pathlib.PurePosixPath(*parts[:i]).as_posix() in file_paths:
                    raise RuntimeError("Конфликт файлового пути в архиве: " + path)

        total_written = 0
        for member in members:
            if member.is_dir():
                target = destination.joinpath(*pathlib.PurePosixPath(member.filename).parts)
                target.mkdir(parents=True, exist_ok=True)
                continue

            target = destination.joinpath(*pathlib.PurePosixPath(member.filename).parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            written = 0
            with archive.open(member, "r") as source, open(target, "xb") as output:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    written += len(chunk)
                    total_written += len(chunk)
                    if written > max_single_file or total_written > max_total_uncompressed:
                        raise RuntimeError("Фактический распакованный размер ZIP превышает лимит")
                    output.write(chunk)
            if written != member.file_size:
                raise RuntimeError("Фактический размер файла не совпадает с ZIP-заголовком: " + member.filename)

print("Выберите Zvukotseh_Studio_Pro_1.0.0_Source.zip")
uploaded = files.upload()
# Colab appends (1), (2), etc. to repeated uploads. Accept these names,
# while SHA-256 below still verifies the exact archive contents.
zip_candidates = [n for n in uploaded if re.fullmatch(
    r"EP133_Mobile_Studio_4\.3\.4_Source(?: \(\d+\))?\.zip", pathlib.Path(n).name
)]
if len(zip_candidates) != 1:
    raise RuntimeError(
        "Выберите ровно один архив Zvukotseh_Studio_Pro_1.0.0_Source.zip. "
        f"Найдено: {len(zip_candidates)}; загружены: {list(uploaded.keys())}"
    )
zip_name = zip_candidates[0]
print("Выбран архив:", zip_name)
actual_sha = hashlib.sha256(pathlib.Path(zip_name).read_bytes()).hexdigest()
if actual_sha != EXPECTED_SHA256:
    raise RuntimeError("SHA-256 не совпадает с комплектом релиза. Ожидался: " + EXPECTED_SHA256 + ", получен: " + actual_sha)
print("SHA-256 исходников подтверждён:", actual_sha)

print("Устанавливаю Java 17 и системные инструменты...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk", "unzip", "wget"], check=True)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["ANDROID_HOME"] = "/content/android-sdk"
os.environ["ANDROID_SDK_ROOT"] = os.environ["ANDROID_HOME"]
os.makedirs(os.environ["ANDROID_HOME"], exist_ok=True)

cmd_zip = "/content/android-commandlinetools.zip"
subprocess.run(["wget", "-q", "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip", "-O", cmd_zip], check=True)
verify_sha256(cmd_zip, ANDROID_TOOLS_SHA256, "Android command-line tools 11076708")
tmp = "/content/android-cmdline-tmp"
shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp, exist_ok=True)
with zipfile.ZipFile(cmd_zip) as z: z.extractall(tmp)
sdk_tools = os.path.join(os.environ["ANDROID_HOME"], "cmdline-tools", "latest")
os.makedirs(os.path.dirname(sdk_tools), exist_ok=True)
shutil.rmtree(sdk_tools, ignore_errors=True); shutil.move(os.path.join(tmp, "cmdline-tools"), sdk_tools)
sdkmanager = os.path.join(sdk_tools, "bin", "sdkmanager")
# Restore executable permissions; ZIP extraction in Colab can drop them.
os.chmod(sdkmanager, os.stat(sdkmanager).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
os.environ["PATH"] += ":" + sdk_tools + "/bin:" + os.environ["ANDROID_HOME"] + "/platform-tools"
subprocess.run(
    [sdkmanager, f"--sdk_root={os.environ['ANDROID_HOME']}", "--licenses"],
    input=("y\n" * 100), text=True, check=True
)
subprocess.run(
    [sdkmanager, f"--sdk_root={os.environ['ANDROID_HOME']}",
     "platform-tools", "platforms;android-35", "build-tools;35.0.0"],
    check=True
)

print("Устанавливаю Gradle 8.9...")
gradle_zip = "/content/gradle-8.9-bin.zip"
subprocess.run(["wget", "-q", "https://services.gradle.org/distributions/gradle-8.9-bin.zip", "-O", gradle_zip], check=True)
verify_sha256(gradle_zip, GRADLE_SHA256, "Gradle 8.9 binary distribution")
with zipfile.ZipFile(gradle_zip) as z: z.extractall("/content")
os.environ["PATH"] += ":/content/gradle-8.9/bin"

project_dir = "/content/ep133-release"
shutil.rmtree(project_dir, ignore_errors=True); os.makedirs(project_dir, exist_ok=True)
safe_extract(zip_name, project_dir)
required_paths = ["app/build.gradle", "app/src/main/AndroidManifest.xml", "tools/verify_release.py"]
missing_paths = [p for p in required_paths if not pathlib.Path(project_dir, p).is_file()]
if missing_paths: raise RuntimeError("В архиве отсутствуют обязательные файлы: " + ", ".join(missing_paths))
manifest = pathlib.Path(project_dir, "app/build.gradle").read_text(encoding="utf-8")
version_code = re.search(r"versionCode\s+(\d+)", manifest)
version_name = re.search(r"versionName\s+['\"]([^'\"]+)['\"]", manifest)
if not version_code or not version_name:
    raise RuntimeError("Не удалось прочитать версию приложения из app/build.gradle")
version_code = version_code.group(1)
version_name = version_name.group(1)
for required in ["minSdk 28", "compileSdk 35"]:
    if required not in manifest: raise RuntimeError("Конфигурация релиза не совпадает: " + required)
print(f"Версия исходников: {version_name} (versionCode {version_code})")
verify_script = os.path.join(project_dir, "tools", "verify_release.py")
if not os.path.isfile(verify_script): raise RuntimeError("Не найден tools/verify_release.py")
subprocess.run(["python3", verify_script], cwd=project_dir, check=True)

print("Собираю APK...")
result = subprocess.run(["/content/gradle-8.9/bin/gradle", "--no-daemon", "--stacktrace", "clean", "assembleDebug"], cwd=project_dir, env=os.environ.copy())
if result.returncode != 0: raise RuntimeError("Gradle завершился с ошибкой. APK не создан.")
apks = glob.glob(project_dir + "/**/app-debug.apk", recursive=True)
if len(apks) != 1: raise RuntimeError(f"Ожидался один APK, найдено: {len(apks)}")
apk = apks[0]
with zipfile.ZipFile(apk) as z:
    bad = z.testzip()
    if bad: raise RuntimeError("APK повреждён: " + bad)
    names = set(z.namelist())
    if "AndroidManifest.xml" not in names or "classes.dex" not in names:
        raise RuntimeError("APK не содержит обязательные Android-компоненты")
    if not any(n.startswith("META-INF/") and n.endswith((".RSA", ".DSA", ".EC")) for n in names):
        raise RuntimeError("В APK не обнаружена debug-подпись")
aapt = os.path.join(os.environ["ANDROID_HOME"], "build-tools", "35.0.0", "aapt")
badging = subprocess.check_output([aapt, "dump", "badging", apk], text=True)
for item in ["package: name='com.ep133.mobile'", f"versionCode='{version_code}'", f"versionName='{version_name}'", "sdkVersion:'28'"]:
    if item not in badging: raise RuntimeError("Метаданные APK не совпали: " + item)
sha_apk = hashlib.sha256(pathlib.Path(apk).read_bytes()).hexdigest()
final_apk = "/content/Zvukotseh_Studio_Pro_1.0.0-debug.apk"
shutil.copy2(apk, final_apk)
print("APK собран. Метаданные и целостность проверены.")
print("Размер MB:", round(os.path.getsize(final_apk)/1024/1024, 2))
print("SHA-256:", sha_apk)
print("Внимание: debug APK. Реальную работу звука, MIDI и записи нужно проверить на телефоне.")
files.download(final_apk)
